# Airflow & Orchestration — Crash Course> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## Fontos megjegyzésAz Apache Airflow-t **nem érdemes** Jupyterben futtatni — valódi scheduler-re és webserver-re van szüksége. Ez a notebook:- a DAG **kódmintát** mutatja be, amit `airflow/dags/` mappába másolhatsz.- Lokális Airflow indítási parancsokat tartalmaz.- A DAG logikát szimuláljuk Pythonban, hogy megértsd a működést.

## 1. Airflow lokálisan DockerrelA legegyszerűbb indítás:```bashmkdir airflow-local && cd airflow-localcurl -LfO 'https://airflow.apache.org/docs/apache-airflow/2.9.0/docker-compose.yaml'mkdir -p ./dags ./logs ./plugins ./configecho "AIRFLOW_UID=50000" > .envdocker compose up airflow-initdocker compose up```Ezután: http://localhost:8080 (user: airflow, pass: airflow).

## 2. DAG minta — webshop ETLÍrd ezt a fájlt az `airflow/dags/webshop_etl.py` helyre:

In [ ]:
DAG_CODE = '''from datetime import datetime, timedeltafrom airflow import DAGfrom airflow.operators.python import PythonOperatorfrom airflow.operators.bash import BashOperatorfrom airflow.sensors.filesystem import FileSensordefault_args = {    "owner": "data-team",    "retries": 2,    "retry_delay": timedelta(minutes=5),    "email_on_failure": False,}with DAG(    dag_id="webshop_etl_daily",    default_args=default_args,    description="WebShop Pro napi bronze/silver/gold ETL",    schedule="0 2 * * *",        # minden nap 02:00    start_date=datetime(2025, 1, 1),    catchup=False,    tags=["webshop", "medallion"],) as dag:    wait_for_source = FileSensor(        task_id="wait_for_source",        filepath="/data/landing/orders_{{ ds }}.csv",        poke_interval=60,        timeout=600,    )    def extract_bronze(**ctx):        print(f"Bronze ingest: {ctx['ds']}")        # itt olvasnád a CSV-t és írnád Delta bronze-be        return {"rows": 12345}    bronze = PythonOperator(task_id="bronze", python_callable=extract_bronze)    def transform_silver(**ctx):        rows = ctx["ti"].xcom_pull(task_ids="bronze")["rows"]        print(f"Silver clean: {rows} sor")        return {"valid_rows": int(rows * 0.98)}    silver = PythonOperator(task_id="silver", python_callable=transform_silver)    gold = BashOperator(        task_id="gold",        bash_command="echo 'Gold KPI frissítve {{ ds }}'",    )    wait_for_source >> bronze >> silver >> gold'''print(DAG_CODE)

## 3. DAG logika szimulációja Pythonban (Airflow nélkül)

In [ ]:
from datetime import datetimeclass Task:    def __init__(self, name, fn): self.name, self.fn, self.done = name, fn, False    def run(self, ctx):        print(f'▶ {self.name}'); result = self.fn(ctx) or {}; self.done = True        print(f'✓ {self.name} — {result}'); return resultdef run_pipeline(tasks, ctx=None):    ctx = ctx or {'ds': datetime.now().strftime('%Y-%m-%d'), 'xcom': {}}    for t in tasks:        res = t.run(ctx)        ctx['xcom'][t.name] = res    print('\n✓ Pipeline kész')# Szimuláljukrun_pipeline([    Task('wait_for_source', lambda c: {'file': f"orders_{c['ds']}.csv", 'found': True}),    Task('bronze',          lambda c: {'rows': 12345}),    Task('silver',          lambda c: {'valid_rows': int(c['xcom']['bronze']['rows'] * 0.98)}),    Task('gold',            lambda c: {'aggregates_built': True}),])

## 4. Airflow CLI — DAG-ok kezelése```bashairflow dags listairflow tasks list webshop_etl_dailyairflow dags trigger webshop_etl_dailyairflow dags backfill -s 2025-01-01 -e 2025-01-07 webshop_etl_daily```

## Következő lépések- Térj vissza a [web-alapú kurzushoz](airflow-orchestration/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*